In [1]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
ARROW_AVAILABLE = True
# pa.PyExtensionType.set_auto_load(True)
file_path = "/Users/liyong.1024/Documents/protect/redline_database/redline_annotation_0904_merge_v4_prompt15_verl_test.parquet"
df = pd.read_parquet(file_path)



In [7]:
print(df[:1]["prompt"].values)

[array([{'content': '你是专业的电商客服', 'role': 'system'},
        {'content': '\n# 目标\n你是专业的电商客服，你需要分析工作日志，判断客服回答是否存在事实性错误。\n\n# 背景和要求\n## 工作日志格式\n- 上文对话。包含了用户和客服的对话历史。\n- 用户问题。\n- 候选知识。包含了由RAG搜索到的多个相关知识，每个知识包含序号、标题、内容。\n- 客服回答。客服对用户问题的回答。\n\n## 评估标准\n请注意不要关心客服回答是否回答了用户问题，只需要关心客服回答中是否存在事实性错误。\n\n### 事实性错误\n事实性错误包括：\n1.商品信息错误。包括：价格、规格、材质、成分、库存状态、生产日期、保质期、包装清单、功能、产品型号、尺寸、防伪查询及正品鉴定政策、能效等级、环保等级、执行标准等与候选知识或上文对话不一致。\n2.活动/促销信息错误。包括：国补、活动时间、参与活动城市、优惠力度/活动内容（满减、折扣、赠品）、参与条件、优惠券链接、限购数量、活动对象、支付方式等与候选知识或上文对话不符。\n3.政策信息错误。包括：售后及质保政策、运费承担方、价保政策、发票开具规则、安装费用与政策流程等与候选知识或上文对话不符。\n4.物流信息错误。包括：发货地、默认快递、预计送达时间、物流进度、包邮规则等与候选知识或上文对话不符。\n5.错误引导咨询平台客服或第三方。包括：错误的引导用户咨询平台客服，或引导联系第三方咨询问题或办理事项。\n6.错误的进行操作或办理类引导。包括：引导用户进行错误的app操作、下单流程、线下办理、商品操作/使用方法等。\n7.推荐竞品或外部平台。主动或被动地提及、推荐非本店铺的商品，或引导用户去其他电商平台、竞品店铺购买。本店铺没有相关配件或搭配使用的商品时，可以进行推荐，但不可直接或隐晦的提及其它店铺或平台的名字。\n\n### 严格一致性场景\n以下是一些需要严格限制一致性的场景：\n1.关键商品信息。指会影响用户购买决策和使用体验的关键信息，包括：尺码推荐、价格、规格、材质、成分、生产日期、保质期、包装清单、产品型号、尺寸、防伪查询、正品鉴定政策、能效等级、环保等级、执行标准。\n2.所有活动/促销信息。\n3.所有政策信息。\n4.所有物流信息。\n

In [10]:
redline_data = df[(df['redline_label'] == 1)]
tp_data = df[(df['answer_label'] == 1) & (df['answer_pred_p13'] == 1)].sample(frac=0.1)
fp_data = df[(df['answer_label'] == 1) & (df['answer_pred_p13'] == 0)].sample(frac=0.5)
new_df = pd.concat([redline_data] + [fp_data] + [tp_data], ignore_index=True)
new_df.to_excel("/Users/liyong.1024/Documents/protect/redline_database/redline_annotation_0904_merge_v3_filter.xlsx")

In [11]:
new_df

,index,send_time,session_id,message_id,bot_id,user_pin,上文对话,用户问,答案,候选知识,...,redline_label,answer_label,reject,source_file,redline_pred_p13,answer_pred_p13,no_rate,yes_rate,llm_response,rand
0,0,2025-08-14 18:43:33.115,df72d1f5ccfac5b4fa3805387787b6ff,A6A9A135-3973-4B4C-B8D5-30309CDA0406,23489,13170971432_p,"""用户: 顾客加入咨询 订单ID:319132805296\\n\n用户: 我想问下\\n\...",这个手表拦截的电话号码怎么看不到啊,"{""answer"":[{""content"":""您好，米兔儿童电话手表7X支持通话白名单功能，...",[1] 有什么功能\n<strong>米兔儿童电话手表7X</strong><br/>支持通...,...,1,0,False,redline_annotation_0815.xlsx,1,0,0,-1,"{'choices': [{'index': 0, 'finish_reason': 'st...",NaN
1,42,2025-08-14 20:08:47.111,5f857dfd-2f96-4105-a7f5-b93cdab5e652,MasterAgentXZ_7e375630ae214163a39a3e1d2807f080,1081088,jd_zfpuvrobtsoh,"""用户: 顾客jd_zfpuvrobtsoh加入咨询""}",如何参加国补？,"{""answer"":[{""content"":""参加国补的具体步骤如下：1. 确认您购买的产品...","""}",...,1,0,True,redline_annotation_0815.xlsx,1,0,0,-1,"{'choices': [{'index': 0, 'finish_reason': 'st...",NaN
2,134,2025-08-14 20:44:32.825,70fb08082185fe4a4913396f2dee4ad4,600D8474-6676-41CC-8AC9-C47E2B0AC4CD,23489,jd_58e84ed2a2308,"""用户: 有没有吸力25立方的\\n\n客服: 米家智能欧式吸油烟灶套装S2（米家智能欧式吸...",P2是最新款吗,"{""answer"":[{""content"":""米家智能净烟机P2是小米的最新款产品之一，具备...","[1] 是否新款\n<p><span style=\\""color: rgb(192, 0,...",...,1,0,True,redline_annotation_0815.xlsx,1,0,0,-1,"{'choices': [{'index': 0, 'finish_reason': 'st...",NaN
3,182,2025-08-14 22:41:20.846,8d0098a83ac0c02674c4f32a94aa5fc9,0cb8dbcf-5257-1ffa-28a5-92daf244a545,132483,jd_4aa5ebdd36aaa,"""用户: 顾客jd_4aa5ebdd36aaa加入咨询，商品编号：100171882477\...",我娃3个多月现在15斤 L会不会小呀 肚子上很多肉肉,"{""answer"":[{""content"":""根据您的描述，L码适合7-12个月，体重9-1...",[1] L码多大宝宝用\n根据您的描述推荐：L码<br/>#E-s01适合月龄：7-12月（...,...,1,0,False,redline_annotation_0815.xlsx,1,0,0,-1,"{'choices': [{'index': 0, 'finish_reason': 'st...",NaN
4,216,2025-08-14 07:57:45.402,6ff12fd4f539d1e4917b19bfb504242c,facf35054cb64fbcb371b681d75ad86c,132,jd_5472e0e9e62eb,"""用户: 顾客jd_5472e0e9e62eb加入咨询 商品：100183021313\\\...",要底座充电吗 支持快充吗,"{""answer"":[{""content"":""vivo WATCH 5手表仅支持通过配置的充...",[1] 供电方式及充电器参数\nvivo WATCH 5手表仅支持<span style=\...,...,1,0,False,redline_annotation_0815.xlsx,1,0,0,-1,"{'choices': [{'index': 0, 'finish_reason': 'st...",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3227,7540,2025-08-25T18:35:01.368000,8901114afca57378b8e0211b3c7e264b,36d2db4556434e548d18c3f1b066d7b6,23489,jd_5ab4f51872e33,"{""formatted_context"": ""用户: 顾客jd_5ab4f51872e33加...",怎么使用,"{""answer"":[{""content"":""米家智能腰部按摩仪EMS版<br/>【使用说明...",[1] 商品使用方法\n米家智能腰部按摩仪EMS版<br/>【使用说明】?【<a href=...,...,0,1,False,redline_annotation_0826.xlsx,0,1,0,-1,"{'choices': [{'index': 0, 'finish_reason': 'st...",0.531141
3228,8279,2025-08-25T06:11:49.396000,c65ac20668b4b310ddb1194a4c5c1e13,72e6ec8543ef48579f8086ecbfb1a23d,23489,jd_wepmiufjulib,"{""formatted_context"": ""用户: 顾客jd_wepmiufjulib加入...",什么是玻纤材质什么是玻纤材质？？,"{""answer"":[{""content"":""玻纤材质是指玻璃纤维增强塑料（Glass Fi...",[1] 名词释义\n商品功能、名词说明可以参考商品详情页哦~\n\n[2] 商品材质\n<s...,...,0,1,False,redline_annotation_0826.xlsx,0,1,0,-1,"{'choices': [{'index': 0, 'finish_reason': 'st...",0.088937
3229,6031,2025-08-21T16:11:04.909000,23dfe947562337fab9935a62684d94e1,6498AA2C-83BD-419B-815F-D77ED259F55F,543317,jd_64a6c25cc9c37,"""用户: https://item.jd.com/10156627110570.html?...",16.8/19.9/59.9什么区别 看外观都一样,"{""answer"":[{""content"":""您好，关于不同款型商品对比，您可以：1、对比详...",[1] 商品区别\n尊敬的顾客您好！感谢信任选择本店。<br/>关于不同款型商品对比，您可以...,...,0,1,False,redline_annotation_0822.xlsx,0,1,0,-1,"{'choices': [{'index': 0, 'finish_reason': 'st...",0.653778
3230,4770,2025-08-21T09:02:26.040000,bcd20308b4f99d72e38e8c1bd7da61a0,bdeeb296-64ad-bf1a-7493-8ee5485a95b3,769654,jd_WDmDDZttisHQ,"""用户: 顾客jd_WDmDDZttisHQ加入咨询，商品编号：1011530652386...",是真品吗,"{""answer"":[{""content"":""店铺所售商品均为正品行货，商品都是通过正规渠道...",[1] 正品保障\n店铺所售商品均为正品行货，商品都是通过正规渠道进货，商品质量都经过严格的...,...,0,1,False,redline_annotation_0822.xlsx,0,1,0,-1,"{'choices': [{'index': 0, 'finish_reason': 'st...",0.017648


In [43]:
import json

request = df.loc[1]["request"]
response = df.loc[1]["response"]
def parse_request(request):
    try:
        # print(request)
        req_json = json.loads(request) 

        # print(req_json)
        return req_json
    except:
        print(f"error:{request}")
        return {}


def parse_response(response):
    try:
        response = f"{response}"
        response = response.replace("\\\\", "\\")
        # print(response)
        resp_json = json.loads(response) 
        print(resp_json)
        resp_json = json.loads(resp_json) 
        print(resp_json["reject"])
        if "reject" not in  resp_json:
             resp_json["reject"] = False
        return resp_json
    except:
        print(f"error:{response}")
        return {"reject":False,"reason":""}

parse_response(response)
parse_request(request)


{"reject":true,"reason":"<think>\n\n</think>\n\n{\"一级\":\"事实错误\",\"二级\": \"尺码问题\"}"}
True


{'answer': [{'content': '', 'type': 'text'}],
 'context': [{'content': '160斤身高1.7 穿多大的号',
   'intent': 'knowledge_answer',
   'role': 'user'},
  {'content': '您好，根据您的身高和体重，建议您选择适合您的尺码。您可以参考店铺内的尺码表，选择最适合您的尺码。如果您有具体的产品型号或系列名称，可以告诉我，我可以为您提供更详细的建议。;',
   'role': 'waiter'},
  {'content': 'https://item.jd.com/10151633846635.html?sdx=ehi-lLxFuZiE6JnJZIJaj8UotDaXCw4rsmpEuKhAY9iCPe_RLJhZ53rsrUrhUmGZ\\t 商品ID：10151633846635，商品名称：YOUNG VERSACE范思新款哲男士夏季新款T恤圆领潮流短袖百搭印花半袖时尚男士 黑色 范思哲 XL 范思哲',
   'role': 'user'},
  {'content': '很高兴为您服务，选择这款商品的您超有眼光，请问您要咨询这款产品什么问题呢？', 'role': 'waiter'}],
 'knowledges': [{'answer': [{'content': '店铺有多款产品供您选择，所有系列都是不错的呢，您可以根据需求在店铺首页搜索选择的哦~',
     'type': 'text'}],
   'id': 1,
   'isPkAnswer': 1,
   'name': '尺码推荐',
   'score': 0.9512712,
   'type': 'faq_nlu_manager'},
  {'answer': [{'content': '您可以点击商品详情查看“规格与包装”里面的配件信息哦~', 'type': 'text'}],
   'id': 2,
   'isPkAnswer': 0,
   'name': '包装清单',
   'score': 0.33268708,
   'type': 'faq_nlu_manager'},
  {'answer': [{'content': '每个人

In [38]:

# 方法3: 使用assign方法（更简洁）
print("方法3: 使用assign方法")
# parsed_data = df['request'].apply(parse_request)
parsed_data2 = df['response'].apply(parse_response)
print(parsed_data2)
df_result3 = df.assign(
    # answer=parsed_data.apply(lambda x: x['answer']),
    # context=parsed_data.apply(lambda x: x['context']),
    # knowledges=parsed_data.apply(lambda x: x['knowledges']),
    # query=parsed_data.apply(lambda x: x['query']),
    reject=parsed_data2.apply(lambda x: x['reject']),
    reason=parsed_data2.apply(lambda x: x['reason']),
)

print(df_result3)


方法3: 使用assign方法
error:nan
0        {"reject":false,"reason":"<think>\n\n</think>\...
1        {"reject":true,"reason":"<think>\n\n</think>\n...
2        {"reject":false,"reason":"<think>\n\n</think>\...
3        {"reject":false,"reason":"<think>\n\n</think>\...
4        {"reject":false,"reason":"<think>\n\n</think>\...
                               ...                        
31424    {"reject":false,"reason":"<think>\n\n</think>\...
31425    {"reject":true,"reason":"<think>\n\n</think>\n...
31426    {"reject":true,"reason":"<think>\n\n</think>\n...
31427    {"reject":false,"reason":"<think>\n\n</think>\...
31428    {"reject":false,"reason":"<think>\n\n</think>\...
Name: response, Length: 31429, dtype: object


TypeError: string indices must be integers, not 'str'

In [ ]:

# 方法3: 使用assign方法（更简洁）
print("方法3: 使用assign方法")
# parsed_data = df['request'].apply(parse_request)
parsed_data2 = df['response'].apply(parse_response)
print(parsed_data2)
df_result3 = df.assign(
    # answer=parsed_data.apply(lambda x: x['answer']),
    # context=parsed_data.apply(lambda x: x['context']),
    # knowledges=parsed_data.apply(lambda x: x['knowledges']),
    # query=parsed_data.apply(lambda x: x['query']),
    reject=parsed_data2.apply(lambda x: x['reject']),
    reason=parsed_data2.apply(lambda x: x['reason']),
)

print(df_result3)


方法3: 使用assign方法
error:nan
0        {"reject":false,"reason":"<think>\n\n</think>\...
1        {"reject":true,"reason":"<think>\n\n</think>\n...
2        {"reject":false,"reason":"<think>\n\n</think>\...
3        {"reject":false,"reason":"<think>\n\n</think>\...
4        {"reject":false,"reason":"<think>\n\n</think>\...
                               ...                        
31424    {"reject":false,"reason":"<think>\n\n</think>\...
31425    {"reject":true,"reason":"<think>\n\n</think>\n...
31426    {"reject":true,"reason":"<think>\n\n</think>\n...
31427    {"reject":false,"reason":"<think>\n\n</think>\...
31428    {"reject":false,"reason":"<think>\n\n</think>\...
Name: response, Length: 31429, dtype: object


TypeError: string indices must be integers, not 'str'

In [ ]:

# 方法3: 使用assign方法（更简洁）
print("方法3: 使用assign方法")
# parsed_data = df['request'].apply(parse_request)
parsed_data2 = df['response'].apply(parse_response)
print(parsed_data2)
df_result3 = df.assign(
    # answer=parsed_data.apply(lambda x: x['answer']),
    # context=parsed_data.apply(lambda x: x['context']),
    # knowledges=parsed_data.apply(lambda x: x['knowledges']),
    # query=parsed_data.apply(lambda x: x['query']),
    reject=parsed_data2.apply(lambda x: x['reject']),
    reason=parsed_data2.apply(lambda x: x['reason']),
)

print(df_result3)


方法3: 使用assign方法
error:nan
0        {"reject":false,"reason":"<think>\n\n</think>\...
1        {"reject":true,"reason":"<think>\n\n</think>\n...
2        {"reject":false,"reason":"<think>\n\n</think>\...
3        {"reject":false,"reason":"<think>\n\n</think>\...
4        {"reject":false,"reason":"<think>\n\n</think>\...
                               ...                        
31424    {"reject":false,"reason":"<think>\n\n</think>\...
31425    {"reject":true,"reason":"<think>\n\n</think>\n...
31426    {"reject":true,"reason":"<think>\n\n</think>\n...
31427    {"reject":false,"reason":"<think>\n\n</think>\...
31428    {"reject":false,"reason":"<think>\n\n</think>\...
Name: response, Length: 31429, dtype: object


TypeError: string indices must be integers, not 'str'